In [1]:
!pip install -q -U google-genai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.


In [2]:
pip install -q -U sentence-transformers pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/5

In [14]:
import pandas as pd
import json
import uuid
import time
from datetime import datetime
from google import genai
from google.genai import types
import torch
import torchvision
import transformers
from sentence_transformers import SentenceTransformer

print(" PyTorch Version:", torch.__version__)
print(" Torchvision Version:", torchvision.__version__)
print(" Transformers Version:", transformers.__version__)
print(" Imports successful! Ready for Phase 2.")

 PyTorch Version: 2.11.0+cu130
 Torchvision Version: 0.25.0+cpu
 Transformers Version: 5.0.0
 Imports successful! Ready for Phase 2.


In [4]:
file_path = "api_ready_medical_dataset.jsonl"
df = pd.read_json(file_path, lines=True)
df.head(10)

,id,domain,english_text,bengali_text,tags,metadata
0,0b4eab11-3d42-461f-a2ee-887ef3206342,Pharmacy dosage and medication instructions,Take one tablet twice daily after meals.,প্রতিদিন খাবারের পর একটি ট্যাবলেট দিনে দুবার স...,"[medication, dosage, frequency, post-meal]","{'created_at': '2026-03-24T15:49:47.700247', '..."
1,4720cc45-7856-46a5-abfd-a854b2dd6575,Pharmacy dosage and medication instructions,Apply a thin layer of cream to the affected ar...,আক্রান্ত স্থানে দিনে তিনবার একটি পাতলা স্তরের ...,"[topical, cream, application, frequency]","{'created_at': '2026-03-24T15:49:47.700324', '..."
2,632a424f-9362-43c1-a604-278e3f4b428c,Pharmacy dosage and medication instructions,Shake the bottle well before each use.,প্রতিটি ব্যবহারের আগে বোতলটি ভালোভাবে ঝাঁকিয়ে ...,"[liquid, suspension, instructions, preparation]","{'created_at': '2026-03-24T15:49:47.700367', '..."
3,832a1855-3e7a-463c-b030-a4fb2f9b2af6,Pharmacy dosage and medication instructions,Dissolve two sachets in a glass of water and d...,দুটি স্যাশে এক গ্লাস পানিতে গুলে তাৎক্ষণিকভাবে...,"[powder, solution, preparation, administration]","{'created_at': '2026-03-24T15:49:47.700399', '..."
4,76cb8e33-4e47-4c92-b623-3d4333722019,Pharmacy dosage and medication instructions,This medication should be taken on an empty st...,"এই ঔষধটি খালি পেটে, সকালের খাবারের ৩০ মিনিট আগ...","[medication, dosage, pre-meal, empty-stomach]","{'created_at': '2026-03-24T15:49:47.700429', '..."
5,dd8a5ef9-537f-4e1f-9951-cde11d1d5ac4,Pharmacy dosage and medication instructions,"Complete the full course of antibiotics, even ...",এমনকি ভালো বোধ করলেও অ্যান্টিবায়োটিকের সম্পূর...,"[antibiotics, course, compliance, duration]","{'created_at': '2026-03-24T15:49:47.700459', '..."
6,c1faf673-a930-49e5-9c3f-adb07054b349,Pharmacy dosage and medication instructions,"For children, administer 5 ml of suspension or...","শিশুদের জন্য, দিনে দুবার মুখে ৫ মিলি সাসপেনশন ...","[pediatric, liquid, dosage, oral]","{'created_at': '2026-03-24T15:49:47.700490', '..."
7,29a5dad1-1b64-4163-be95-075a189baf85,Pharmacy dosage and medication instructions,Insert one suppository rectally every night be...,প্রতিদিন রাতে ঘুমানোর আগে একটি সাপোজিটরি মলদ্ব...,"[suppository, rectal, administration, frequency]","{'created_at': '2026-03-24T15:49:47.700522', '..."
8,dc8ca103-245a-4d93-ae49-41f037a7b688,Pharmacy dosage and medication instructions,Do not crush or chew this extended-release tab...,এই এক্সটেন্ডেড-রিলিজ ট্যাবলেটটি চূর্ণ বা চিবাব...,"[tablet, administration, caution, extended-rel...","{'created_at': '2026-03-24T15:49:47.700553', '..."
9,e44484fb-0a44-4c7e-a5a4-14351348ac62,Pharmacy dosage and medication instructions,"Store this medicine in a cool, dry place, away...","এই ঔষধটি ঠাণ্ডা ও শুষ্ক স্থানে, সরাসরি সূর্যাল...","[storage, instructions, caution, temperature]","{'created_at': '2026-03-24T15:49:47.700581', '..."


In [5]:
df['domain'].value_counts()

domain
Pharmacy dosage and medication instructions    100
Emergency room triage questions                100
Post-operative home care instructions          100
Pediatric fever and symptom management         100
Maternal health and pregnancy guidelines       100
Name: count, dtype: int64

In [6]:
# Let's see all the sentences that mention "Paracetamol" or "ট্যাবলেট" (tablet)
df[df['english_text'].str.contains("Paracetamol", case=False)]

,id,domain,english_text,bengali_text,tags,metadata
301,03473395-19fe-483e-8d32-a4c9d694a78d,Pediatric fever and symptom management,Please give your child 5 ml of paracetamol syr...,আপনার সন্তানকে প্রতি ছয় ঘণ্টা অন্তর ৫ মিলি প্...,"[medication, paracetamol, syrup, dosage, fever...","{'created_at': '2026-03-24T15:54:11.328255', '..."
318,330d061a-3ba9-4677-b82d-1af677d08df4,Pediatric fever and symptom management,The doctor recommended alternating between par...,ডাক্তার জ্বর নিয়ন্ত্রণের জন্য প্যারাসিটামল এব...,"[medication strategy, alternating medication, ...","{'created_at': '2026-03-24T15:54:11.330007', '..."
322,f2f28868-1fec-4237-9b4e-8885c36838dc,Pediatric fever and symptom management,Give paracetamol every six hours if needed for...,জ্বরের জন্য প্রয়োজনে প্রতি ছয় ঘণ্টা পর পর প্...,"[medication, paracetamol, dosage, fever manage...","{'created_at': '2026-03-24T15:54:29.209600', '..."
341,6a7e2a11-5b68-4694-991d-fc8c2dbda356,Pediatric fever and symptom management,Please give your child a paracetamol suspensio...,আপনার সন্তানকে ওজন অনুযায়ী প্যারাসিটামল সাসপে...,"[medication, paracetamol, dosage, pediatric, i...","{'created_at': '2026-03-24T15:54:48.158238', '..."
362,38f7b444-7028-428a-b930-911f8b30f683,Pediatric fever and symptom management,Give your child paracetamol according to their...,আপনার শিশুকে তার ওজন অনুযায়ী প্যারাসিটামল দিন।,"[medication, paracetamol, dosage, child]","{'created_at': '2026-03-24T15:55:06.223747', '..."
371,2a387579-8e5d-4b52-9a8e-8efb104d5662,Pediatric fever and symptom management,Administer ibuprofen if paracetamol is not eff...,"যদি প্যারাসিটামল কার্যকর না হয়, তবে আইবুপ্রোফ...","[medication, ibuprofen, paracetamol, consultat...","{'created_at': '2026-03-24T15:55:06.224553', '..."


In [7]:
# Extract the quality status into a new main column
df['quality_status'] = df['metadata'].apply(lambda x: x.get('quality_status'))
print(df['quality_status'].value_counts())

quality_status
unverified    500
Name: count, dtype: int64


In [9]:
import torch.nn.functional as F

In [15]:
print("Loading Dataset and Models...")

input_file = "api_ready_medical_dataset.jsonl"
df = pd.read_json(input_file, lines=True)
initial_count = len(df)
print(f"Loaded {initial_count} unverified translations.")

#Load the Embedding Models

# Model 1: Fast English model for Deduplication
eng_model = SentenceTransformer('all-MiniLM-L6-v2')
# Model 2: Multilingual model for English-to-Bengali Alignment
multi_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')


# FILTER 1: Deduplication (English-to-English)

print("\n Running Filter 1: Deduplication...")
english_texts = df['english_text'].tolist()
eng_embeddings = eng_model.encode(english_texts, convert_to_tensor=True)

# Calculate the cosine similarity matrix for all English sentences
cosine_matrix = F.cosine_similarity(eng_embeddings.unsqueeze(1), eng_embeddings.unsqueeze(0), dim=2)

# Find pairs with similarity > 0.90 (ignoring the diagonal where a sentence matches itself)
duplicates_to_drop = set()
threshold_dedup = 0.90

for i in range(len(cosine_matrix)):
    for j in range(i + 1, len(cosine_matrix)):
        if cosine_matrix[i][j].item() > threshold_dedup:
            duplicates_to_drop.add(j) # Mark the duplicate for deletion

# Drop the duplicates from the DataFrame
df = df.drop(index=list(duplicates_to_drop)).reset_index(drop=True)
print(f"Dropped {len(duplicates_to_drop)} redundant/duplicate English sentences.")


# FILTER 2: Cross-Lingual Alignment

print(" Running Filter 2: Cross-Lingual Alignment...")

# Re-extract texts since we dropped some rows
english_texts = df['english_text'].tolist()
bengali_texts = df['bengali_text'].tolist()

# Embed both languages into the SAME mathematical vector space
eng_multi_embs = multi_model.encode(english_texts, convert_to_tensor=True)
ben_multi_embs = multi_model.encode(bengali_texts, convert_to_tensor=True)

# Calculate 1-to-1 Cosine Similarity between English and its Bengali translation
alignment_scores = F.cosine_similarity(eng_multi_embs, ben_multi_embs)

# Add the scores directly to our DataFrame so we can see the math
df['alignment_score'] = alignment_scores.cpu().numpy()

# Filter out bad translations (Score < 0.75 means the LLM likely hallucinated)
threshold_align = 0.75

# Look at the statistical distribution of the scores
print(df['alignment_score'].describe())

# Look at 5 random sentences that scored between 0.60 and 0.70
mid_tier = df[(df['alignment_score'] >= 0.60) & (df['alignment_score'] < 0.70)]
for index, row in mid_tier.head().iterrows():
    print(f"Score: {row['alignment_score']:.2f}")
    print(f"EN: {row['english_text']}")
    print(f"BN: {row['bengali_text']}\n")
bad_translations = df[df['alignment_score'] < threshold_align]
df_clean = df[df['alignment_score'] >= threshold_align].copy()

print(f"🗑️ Dropped {len(bad_translations)} poor translations (Alignment Score < {threshold_align}).")


# FINALIZE AND SAVE
# Update the metadata to show this data is now mathematically verified
def update_metadata(row):
    meta = row['metadata']
    meta['quality_status'] = "math_verified"
    meta['alignment_score'] = round(float(row['alignment_score']), 4)
    return meta

df_clean['metadata'] = df_clean.apply(update_metadata, axis=1)

# Drop the temporary column and save back to JSONL
df_clean = df_clean.drop(columns=['alignment_score'])
output_file = "verified_medical_dataset.jsonl"
df_clean.to_json(output_file, orient='records', lines=True, force_ascii=False)

final_count = len(df_clean)
print(f"\n Phase 2 Complete! Saved {final_count} verified translations to {output_file}")
print(f" Total data removed: {initial_count - final_count} pairs.")

Loading Dataset and Models...
Loaded 500 unverified translations.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Running Filter 1: Deduplication...
Dropped 151 redundant/duplicate English sentences.
 Running Filter 2: Cross-Lingual Alignment...
count    349.000000
mean       0.310901
std        0.186215
min       -0.000564
25%        0.163907
50%        0.272676
75%        0.434122
max        0.806960
Name: alignment_score, dtype: float64
Score: 0.66
EN: Dissolve two sachets in a glass of water and drink immediately.
BN: দুটি স্যাশে এক গ্লাস পানিতে গুলে তাৎক্ষণিকভাবে পান করুন।

Score: 0.66
EN: Discard any unused medication after 28 days of opening.
BN: খোলার ২৮ দিন পর অব্যবহৃত ঔষধ ফেলে দিন।

Score: 0.68
EN: Take 2 tablets with a full glass of water.
BN: এক গ্লাস ভর্তি পানি দিয়ে দুটি ট্যাবলেট সেবন করুন।

Score: 0.66
EN: Shake well before use.
BN: ব্যবহারের পূর্বে ভালোভাবে ঝাঁকিয়ে নিন।

Score: 0.64
EN: For external use only.
BN: শুধুমাত্র বাহ্যিক ব্যবহারের জন্য।

🗑️ Dropped 341 poor translations (Alignment Score < 0.75).

 Phase 2 Complete! Saved 8 verified translations to verified_medical_data

In [13]:
print("Loading dataset and initializing models...")
input_file = "api_ready_medical_dataset.jsonl"
df = pd.read_json(input_file, lines=True)
print(f"Loaded {len(df)} initial records.")

# Initialize embedding models
eng_model = SentenceTransformer('all-MiniLM-L6-v2')
multi_model = SentenceTransformer('sentence-transformers/LaBSE')

# Filter 1: Deduplication
print("Running deduplication...")
english_texts = df['english_text'].tolist()
eng_embeddings = eng_model.encode(english_texts, convert_to_tensor=True)

cosine_matrix = F.cosine_similarity(eng_embeddings.unsqueeze(1), eng_embeddings.unsqueeze(0), dim=2)
duplicates_to_drop = set()
threshold_dedup = 0.90

for i in range(len(cosine_matrix)):
    for j in range(i + 1, len(cosine_matrix)):
        if cosine_matrix[i][j].item() > threshold_dedup:
            duplicates_to_drop.add(j)

df = df.drop(index=list(duplicates_to_drop)).reset_index(drop=True)
print(f"Dropped {len(duplicates_to_drop)} duplicate sentences.")

# Filter 2: Cross-Lingual Alignment
print("Running cross-lingual alignment scoring...")
english_texts = df['english_text'].tolist()
bengali_texts = df['bengali_text'].tolist()

eng_multi_embs = multi_model.encode(english_texts, convert_to_tensor=True)
ben_multi_embs = multi_model.encode(bengali_texts, convert_to_tensor=True)

alignment_scores = F.cosine_similarity(eng_multi_embs, ben_multi_embs)
df['alignment_score'] = alignment_scores.cpu().numpy()

threshold_align = 0.70
bad_translations = df[df['alignment_score'] < threshold_align]
df_clean = df[df['alignment_score'] >= threshold_align].copy()

print(f"Dropped {len(bad_translations)} records below alignment threshold ({threshold_align}).")

# Update metadata and save
def update_metadata(row):
    meta = row['metadata']
    meta['quality_status'] = "math_verified"
    meta['alignment_score'] = round(float(row['alignment_score']), 4)
    return meta

df_clean['metadata'] = df_clean.apply(update_metadata, axis=1)
df_clean = df_clean.drop(columns=['alignment_score'])

output_file = "verified_medical_dataset_labse.jsonl"
df_clean.to_json(output_file, orient='records', lines=True, force_ascii=False)

print(f"Processing complete. Saved {len(df_clean)} verified pairs to {output_file}")

Loading dataset and initializing models...
Loaded 500 initial records.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running deduplication...
Dropped 151 duplicate sentences.
Running cross-lingual alignment scoring...
Dropped 2 records below alignment threshold (0.7).
Processing complete. Saved 347 verified pairs to verified_medical_dataset_labse.jsonl
